<a href="https://colab.research.google.com/github/moalhadiri/BraudeauRichardsFEM-article/blob/main/run_examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Braudeau–Richards examples

Choose your meshes, final times and time-step levels in the form, then select **Runtime → Run all**. The calculations and figures appear here. No project folder or source-file editing is needed.

Use a CPU runtime. Colab may ask you to sign in with Google. Additional settings for spatial validation and the physical examples are grouped below the main form.


## 1. Choose the common parameters

Changing these settings updates all examples, figures and the README in this session after **Runtime → Run all**. For each dimension, spatial convergence uses three grids ending at your chosen mesh. Time steps follow $\tau=0.1\times2^{1-\ell}$. The fixed-step examples use the smallest selected time step; dry infiltration keeps its adaptive steps. The public GitHub page changes only when the repository owner publishes new results.


In [ ]:
# @title Common settings for all examples
# @markdown Mesh values count subdivisions: n = 13 means h = 1/13. Use at least 4.
mesh_2d = 13 # @param {type:"integer"}
mesh_3d = 13 # @param {type:"integer"}
final_time = 1 # @param {type:"number"}
time_levels = "1, 2, 3" # @param {type:"string"}
use_separate_settings = False # @param {type:"boolean"}
# @markdown All examples use these meshes and T. Spatial curves use three grids ending at n.
# @markdown Sphere and Fourier cases use the smallest selected tau; dry infiltration keeps adaptive time steps.
# @markdown Choose Runtime > Run all after editing. Select use_separate_settings for independent settings below.
calculation_succeeded = False


### Additional settings (optional)

These values are used only when **use_separate_settings** is selected above. Otherwise all examples follow the common settings automatically.


In [ ]:
# @title Separate settings (optional)
# @markdown **Spatial validation.** Comma-separated subdivision counts; the polynomial reference is stationary.
spatial_meshes_2d = "4, 8, 16" # @param {type:"string"}
spatial_meshes_3d = "4, 8" # @param {type:"string"}
spatial_time_level = 3 # @param {type:"integer"}
# @markdown **Slotted sphere.** Times are in hours; n subdivisions give h/L = 1/n.
sphere_mesh = 16 # @param {type:"integer"}
sphere_final_time = 0.1 # @param {type:"number"}
sphere_time_level = 2 # @param {type:"integer"}
# @markdown **Nonlinear Fourier example.** Its constitutive exponent ell_p stays equal to 3.
fourier_mesh = 8 # @param {type:"integer"}
fourier_final_time = 0.2 # @param {type:"number"}
fourier_time_level = 2 # @param {type:"integer"}
# @markdown **Dry infiltration.** Times are in hours; this case keeps adaptive time steps.
dry_mesh_x = 20 # @param {type:"integer"}
dry_mesh_z = 10 # @param {type:"integer"}
dry_final_time = 1.0 # @param {type:"number"}


## 2. Prepare the example

This step prepares the runtime automatically. No Google Drive access is needed.


In [ ]:
from io import BytesIO
from pathlib import Path
import os
import stat
import subprocess
import sys
import tempfile
import urllib.request
import zipfile

# Keep the checked numerical source for repeatable runs.
revision = "3ddfda373130f81eacbad71b1ee98b777550433e"
cache = Path(tempfile.gettempdir()) / ("braudeau_richards_article_" + revision[:8])
project = cache / ("BraudeauRichardsFEM-article-" + revision)
required = (project / "test.py", project / "requirements.txt", project / "parameters.json", project / "source", project / "data")
if not all(path.exists() for path in required):
    cache.mkdir(parents=True, exist_ok=True)
    url = "https://codeload.github.com/moalhadiri/BraudeauRichardsFEM-article/zip/" + revision
    request = urllib.request.Request(url, headers={"User-Agent": "BraudeauRichardsFEM-example"})
    with urllib.request.urlopen(request, timeout=60) as response:
        archive_bytes = response.read()
    with zipfile.ZipFile(BytesIO(archive_bytes)) as archive:
        for entry in archive.infolist():
            destination = (cache / entry.filename).resolve()
            if "\\" in entry.filename or ".." in Path(entry.filename).parts or not destination.is_relative_to(cache.resolve()) or stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError("Unexpected path in the source archive")
        archive.extractall(cache)
if not all(path.exists() for path in required):
    raise RuntimeError("The source archive is incomplete")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-r", str(project / "requirements.txt")], check=True)
print("Ready: source revision", revision[:8])


In [ ]:
# @title Check the selected settings
import json

calculation_succeeded = False

def integer_list(text):
    try:
        return [int(value.strip()) for value in text.split(",") if value.strip()]
    except (AttributeError, ValueError) as error:
        raise ValueError("Enter a comma-separated list of integers") from error

if str(project) not in sys.path:
    sys.path.insert(0, str(project))
from source.utils.configuration import automatic_parameters

if use_separate_settings:
    parameters = json.loads((project / "parameters.json").read_text(encoding="utf8"))
    parameters["spatial"].update(meshes_2d=integer_list(spatial_meshes_2d), meshes_3d=integer_list(spatial_meshes_3d), ell=spatial_time_level, final_time=final_time)
    parameters["temporal"].update(mesh_2d=mesh_2d, mesh_3d=mesh_3d, ell=integer_list(time_levels), final_time=final_time)
    parameters["sphere"].update(mesh=sphere_mesh, ell=sphere_time_level, final_time=sphere_final_time)
    parameters["fourier"].update(mesh=fourier_mesh, ell=fourier_time_level, final_time=fourier_final_time)
    parameters["dry"].update(mesh_x=dry_mesh_x, mesh_z=dry_mesh_z, final_time=dry_final_time)
else:
    parameters = automatic_parameters(mesh_2d, mesh_3d, final_time, integer_list(time_levels))
parameters_path = project / "reviewer_parameters.json"
parameters_path.write_text(json.dumps(parameters, indent=2) + "\n", encoding="utf8", newline="\n")
check = subprocess.run([sys.executable, str(project / "test.py"), "--parameters", str(parameters_path), "--check-parameters"], cwd=project, capture_output=True, text=True, encoding="utf8")
if check.returncode:
    raise ValueError(check.stderr.strip().splitlines()[-1])
checked = json.loads(check.stdout)
print("Selected temporal time steps:")
for ell, tau in checked["time_steps"]["temporal"].items():
    print(f"  ell = {ell}: tau = {tau:g}")
print("Selected meshes and final times:")
for name, values in checked["parameters"].items():
    print("  " + name + ": " + ", ".join(f"{key}={value}" for key, value in values.items()))
print("Dry infiltration keeps its adaptive time steps. Ready to calculate.")


## 3. Run the calculations

The form determines the meshes and final times. Each time-step index gives $\tau=0.1\times2^{1-\ell}$. Here $\ell$ is a time-step index; the Fourier constitutive exponent $\ell_p=3$ is unchanged.

The full numerical output appears below. Each run uses a fresh results folder. Change `workers` below only if you want another number of assembly workers.


In [ ]:
workers = 1
calculation_succeeded = False
results = project / "results"
results.mkdir(exist_ok=True)
output_dir = Path(tempfile.mkdtemp(prefix="run_", dir=results))
command = [sys.executable, "-u", str(project / "test.py"), "--no-show", "--update-readme", "--parameters", str(parameters_path), "--workers", str(workers), "--output", str(output_dir)]
run_environment = dict(os.environ, BRAUDEAU_SOURCE_REVISION=revision)
process = subprocess.Popen(command, cwd=project, env=run_environment, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf8", bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
except KeyboardInterrupt:
    process.terminate()
    process.wait()
    raise
if return_code != 0:
    raise RuntimeError("A calculation or check failed; see the output above")
calculation_succeeded = True


## Results from this run

The table below contains the results just computed. This session updates its own README; the public GitHub page stays unchanged. Select **show_run_readme** to display the full updated README without repeating the calculations. The optional results download also includes this README, its sphere figure and numerical data.


In [ ]:
# @title Results and updated README
show_run_readme = False # @param {type:"boolean"}
from IPython.display import Markdown, display
import base64
import re
import shutil

if not calculation_succeeded:
    raise RuntimeError("Run the calculations successfully before showing their results")
readme = (project / "README.md").read_text(encoding="utf8")
section = readme.split("<!-- BEGINEXAMPLERESULTS -->", 1)[1].split("<!-- ENDEXAMPLERESULTS -->", 1)[0]
table = "\n".join(line for line in section.splitlines() if line.startswith("|"))
display(Markdown(table))
shutil.copyfile(project / "README.md", output_dir / "README.md")
for filename in ("latest_results.json", "latest_results.csv"):
    shutil.copyfile(project / "data" / filename, output_dir / filename)
    (output_dir / "data").mkdir(exist_ok=True)
    shutil.copyfile(project / "data" / filename, output_dir / "data" / filename)
(output_dir / "images").mkdir(exist_ok=True)
shutil.copyfile(output_dir / "slotted_sphere.png", output_dir / "images/slotted_sphere.png")
for filename in ("parameters.json", "CITATION.cff"):
    shutil.copyfile(project / filename, output_dir / filename)
shutil.copyfile(parameters_path, output_dir / "reviewer_parameters.json")
snapshot = json.loads((project / "data/latest_results.json").read_text(encoding="utf8"))
print(f"All {len(snapshot['cases'])} cases are also saved in latest_results.json and latest_results.csv.")
print("Updated README:", output_dir / "README.md")
if show_run_readme:
    picture = base64.b64encode((output_dir / "images/slotted_sphere.png").read_bytes()).decode("ascii")
    preview = readme.replace('src="images/slotted_sphere.png"', 'src="data:image/png;base64,' + picture + '"')
    preview = re.sub(r"\[([^\]]+)\]\(((?:data/[^)]+|parameters\.json|CITATION\.cff))\)", r"\1 (`\2`)", preview)
    display(Markdown(preview))
    print("Linked run files are included in the results ZIP below.")


## 4. View the figures

These images come from the fields just computed. They are also saved as PNG files beside `results.txt`.


In [ ]:
from IPython.display import Image, Markdown, display

if not calculation_succeeded:
    raise RuntimeError("Run the calculations successfully before showing their figures")
figures = [
    ("Spatial errors in 2D", "spatial_errors_2d.png"),
    ("Spatial errors in 3D", "spatial_errors_3d.png"),
    ("Temporal errors", "temporal_errors.png"),
    ("Computed 2D solution", "solution_2d.png"),
    ("Slotted wet sphere in 3D", "slotted_sphere.png"),
    ("Dry infiltration in 2D", "dry_infiltration.png"),
    ("Newton corrections", "newton_corrections.png"),
]
for title, filename in figures:
    path = output_dir / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    display(Markdown("### " + title))
    display(Image(filename=str(path), width=850))
print("Text summary:", output_dir / "results.txt")


## Optional: save the results

Select **download_results** and run this cell to download the updated README, text, seven figures and JSON/CSV data together. The README image and data links also work after extracting the ZIP. This download does not change the public GitHub repository.


In [ ]:
# @title Download this run
download_results = False # @param {type:"boolean"}

if download_results:
    import shutil
    if not calculation_succeeded:
        raise RuntimeError("No successful calculation is available to download")
    archive = shutil.make_archive(str(output_dir), "zip", root_dir=output_dir)
    try:
        from google.colab import files
    except ImportError:
        from IPython.display import FileLink
        export_folder = Path.cwd() / "results"
        export_folder.mkdir(exist_ok=True)
        export = export_folder / "braudeau_results.zip"
        shutil.copyfile(archive, export)
        display(FileLink("results/" + export.name))
    else:
        files.download(archive)
